# Exercise 5.2: Local Evaluation — Error Analysis

This notebook performs systematic error analysis and local evaluation
of our best model (Experiment O: ModernBERT-large verbalizer) on the
official dev set.

**Contents:**
1. Error Analysis — manual inspection of failure modes (FN/FP)
2. Baseline comparison — cross-tabulate our model vs baseline predictions
3. Confusion matrix & precision–recall curves
4. Ablation: threshold sensitivity analysis
5. Per-keyword / per-category breakdown
6. Error length & confidence analysis

In [ ]:
import os
import sys
import json
import logging
import textwrap
import gc

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, roc_curve, roc_auc_score,
)
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from utils.data import load_data_with_keyword, load_data_categories, PCL_CATEGORIES
from utils.pcl_dataset import PCLVerbalizerDataset, PCLDataset
from utils.pcl_deberta import PCLVerbalizer, PCLDeBERTa, PoolingStrategy
from utils.feature_comb import FeatureComb
from utils.eval import evaluate
from utils.feature import extract_keyword_feature

SEED = 42
DATA_DIR = "data"
OUT_DIR = "out"
MODEL_OUT_DIR = OUT_DIR
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================================================================
# PRIMARY MODEL — the model being analysed
# ===================================================================
EXP_NAME = "O_verbalizer"
MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 256
BATCH_SIZE = 32

# ===================================================================
# BASELINES — edit these to swap baselines easily
# Each baseline needs: exp_name, model_name, model_type
# For classifier baselines, also set params_dir and model_dir
# ===================================================================
BASELINE_CLASSIFIER_EXP = "A_keyword"                        # ← EDIT to change classifier baseline
BASELINE_CLASSIFIER_MODEL = "microsoft/deberta-v3-base"       # ← model name for classifier baseline
BASELINE_CLASSIFIER_PARAMS_DIR = "exp/out"                    # ← where best_params.json lives
BASELINE_CLASSIFIER_MODEL_DIR = "exp/out"                     # ← where best_model.pt lives (overridden below for lab)

BASELINE_VERBALIZER_EXP = "L_verbalizer"                      # ← EDIT to change verbalizer baseline
BASELINE_VERBALIZER_MODEL = "roberta-large"                   # ← model name for verbalizer baseline
BASELINE_VERBALIZER_PARAMS_DIR = "exp/out"                    # ← where best_params.json lives
BASELINE_VERBALIZER_MODEL_DIR = "exp/out"                     # ← where best_model.pt lives (overridden below for lab)
# ===================================================================

torch.manual_seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")

if os.getlogin() == "jc4423":
    LOG.info("Running on lab machines, changing caches and model out dir")
    os.environ["HF_HOME"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["TRANSFORMERS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["HF_DATASETS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["PIP_CACHE_DIR"] = "/vol/bitbucket/jc4423/.cache/pip"
    os.environ["XDG_CACHE_HOME"] = "/vol/bitbucket/jc4423/.cache"
    MODEL_OUT_DIR = "/vol/bitbucket/jc4423/models/"
    BASELINE_CLASSIFIER_MODEL_DIR = "exp/out"       # exp_A saves to local out/
    BASELINE_VERBALIZER_MODEL_DIR = "/vol/bitbucket/jc4423/models/"

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load Model & Generate Predictions

In [ ]:
# --- Load best config ---
with open(os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_params.json")) as f:
    best_config = json.load(f)

BEST_THRESHOLD = best_config["best_threshold"]
TEMPLATE_IDX = best_config["template_idx"]
VERBALIZER_NAME = best_config["verbalizer"]

# --- Verbalizer setup ---
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

VERBALIZER_SETS = [
    ("Yes_No",     ["Yes"],  ["No"]),
    ("yes_no",     ["yes"],  ["no"]),
    ("True_False", ["True"], ["False"]),
    ("true_false", ["true"], ["false"]),
]

TEMPLATE_OPTIONS = [
    'Is the following text patronising or condescending? "{text}" Answer: {mask}',
    'Does the author of this text sound patronising or condescending? "{text}" Answer: {mask}',
    'Is this text talking down to people? "{text}" Answer: {mask}',
    'Does the following text contain patronising language? "{text}" Answer: {mask}',
]

verb_set = [v for v in VERBALIZER_SETS if v[0] == VERBALIZER_NAME][0]
pos_ids = [tokeniser.encode(w, add_special_tokens=False)[0] for w in verb_set[1]]
neg_ids = [tokeniser.encode(w, add_special_tokens=False)[0] for w in verb_set[2]]
template = TEMPLATE_OPTIONS[TEMPLATE_IDX]

print(f"Verbalizer: {VERBALIZER_NAME}")
print(f"Template:   {template}")
print(f"Threshold:  {BEST_THRESHOLD}")

In [ ]:
# --- Load model ---
model = PCLVerbalizer(
    pos_verbalizer_ids=pos_ids,
    neg_verbalizer_ids=neg_ids,
    model_name=MODEL_NAME,
    gradient_checkpointing=False,
    cache_dir="/vol/bitbucket/jc4423/.cache/huggingface" if os.getlogin() == "jc4423" else None,
).to(DEVICE)

state_dict = torch.load(
    os.path.join(MODEL_OUT_DIR, f"exp_{EXP_NAME}_best_model.pt"),
    map_location=DEVICE,
)
model.load_state_dict(state_dict)
model.eval()
LOG.info("Model loaded.")

In [ ]:
# --- Load dev data (with keyword + categories for later analysis) ---
_, dev_kw_df = load_data_with_keyword(DATA_DIR)
_, dev_cat_df = load_data_categories(DATA_DIR)

# Merge keyword + categories into one frame
dev_df = dev_kw_df.copy()
for cat in PCL_CATEGORIES:
    dev_df[cat] = dev_cat_df[cat]

LOG.info(f"Dev set: {len(dev_df)} samples, {dev_df['binary_label'].sum()} positive")

# --- Create dataloader ---
dev_ds = PCLVerbalizerDataset(
    texts=dev_df["text"].tolist(),
    labels=dev_df["binary_label"].tolist(),
    max_length=MAX_LENGTH,
    tokeniser=tokeniser,
    template=template,
)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# --- Run inference (collect raw scores for PR curve) ---
model.eval()
all_scores = []
all_labels = []
with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        mask_token_indices = batch["mask_token_indices"].to(DEVICE)
        labels = batch["labels"]

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_token_indices=mask_token_indices,
        ).squeeze(-1)
        all_scores.append(logits.cpu())
        all_labels.append(labels)

all_scores = torch.cat(all_scores)
all_labels = torch.cat(all_labels)
probs = torch.sigmoid(all_scores).numpy()
labels_np = all_labels.long().numpy()
preds_np = (probs >= BEST_THRESHOLD).astype(int)

# Attach predictions & scores to the DataFrame
dev_df = dev_df.copy()
dev_df["pred"] = preds_np
dev_df["score"] = probs
dev_df["correct"] = (dev_df["pred"] == dev_df["binary_label"]).astype(int)

# Classification buckets
dev_df["error_type"] = "TN"
dev_df.loc[(dev_df["binary_label"] == 1) & (dev_df["pred"] == 1), "error_type"] = "TP"
dev_df.loc[(dev_df["binary_label"] == 1) & (dev_df["pred"] == 0), "error_type"] = "FN"
dev_df.loc[(dev_df["binary_label"] == 0) & (dev_df["pred"] == 1), "error_type"] = "FP"

print(dev_df["error_type"].value_counts())
print(f"\nF1: {f1_score(labels_np, preds_np):.4f}")

## 2. Error Analysis — Manual Inspection

Inspect false positives and false negatives to understand failure modes.

In [ ]:
def show_samples(df: pd.DataFrame, title: str, n: int = 10, sort_col: str = "score",
                 ascending: bool = True):
    """Pretty-print samples with their text, score, and metadata."""
    subset = df.sort_values(sort_col, ascending=ascending).head(n)
    print(f"\n{'='*80}")
    print(f" {title} (n={len(df)}, showing {min(n, len(df))})")
    print(f"{'='*80}")
    for i, (idx, row) in enumerate(subset.iterrows()):
        wrapped = textwrap.fill(row['text'], width=90, initial_indent='  ', subsequent_indent='  ')
        print(f"\n[{i+1}] par_id={idx} | keyword={row['keyword']} | "
              f"score={row['score']:.4f} | label={row['binary_label']}")
        print(wrapped)
    print()

In [ ]:
fn_df = dev_df[dev_df["error_type"] == "FN"]
fp_df = dev_df[dev_df["error_type"] == "FP"]

# False Negatives: PCL samples the model missed (sorted by highest confidence wrong)
show_samples(fn_df, "FALSE NEGATIVES — PCL missed by model", n=15, sort_col="score", ascending=True)

# False Positives: Non-PCL samples the model incorrectly flagged (sorted by highest confidence wrong)
show_samples(fp_df, "FALSE POSITIVES — Non-PCL flagged by model", n=15, sort_col="score", ascending=False)

### 2.1 Observations

Manually inspect the samples above and note patterns. Edit the cells below
with your observations.

**False Negatives (missed PCL):**
- *TODO: Note patterns — e.g. sarcasm, subtle condescension, specific topics*

**False Positives (wrongly flagged):**
- *TODO: Note patterns — e.g. emotive language without condescension, keywords triggering*

## 3. Baseline Comparison

Compare our model's predictions against two baselines:
1. **Classifier baseline** (exp_A): `PCLDeBERTa` with keyword features
2. **Verbalizer baseline** (exp_L): `PCLVerbalizer` on `roberta-large`

This reveals cases where models agree (both right / both wrong) and disagree.

To swap baselines, edit the `BASELINE_*` variables in the setup cell (cell 2).

In [ ]:
# ===================================================================
# Helper: generate predictions from a baseline model
# ===================================================================
POOLING_MAP = {
    "cls": PoolingStrategy.CLS, "mean": PoolingStrategy.MEAN,
    "max": PoolingStrategy.MAX, "cls_mean": PoolingStrategy.CLS_MEAN,
}

def load_baseline_config(params_dir: str, exp_name: str) -> dict:
    path = os.path.join(params_dir, f"exp_{exp_name}_best_params.json")
    with open(path) as f:
        return json.load(f)


@torch.no_grad()
def generate_classifier_baseline_preds(
    exp_name: str, model_name: str, params_dir: str, model_dir: str,
    dev_kw_df: pd.DataFrame,
) -> np.ndarray:
    """Load a PCLDeBERTa classifier baseline and return dev predictions."""
    cfg = load_baseline_config(params_dir, exp_name)
    threshold = cfg["best_threshold"]
    tokeniser_bl = AutoTokenizer.from_pretrained(model_name)

    # Rebuild keyword vocab from config
    kw_vocab = cfg["keyword_vocab"]
    kw_to_idx = {kw: i for i, kw in enumerate(kw_vocab)}
    n_extra = len(kw_vocab)

    pooling = POOLING_MAP.get(cfg.get("pooling", "mean"), PoolingStrategy.MEAN)
    feat_comb = FeatureComb.GMF if cfg.get("feature_comb_method") == "GMF" else FeatureComb.CONCAT

    model_bl = PCLDeBERTa(
        hidden_dim=cfg.get("hidden_dim", 256),
        dropout_rate=cfg.get("dropout_rate", 0.0),
        n_extra_features=n_extra,
        pooling=pooling,
        feature_comb_method=feat_comb,
        model_name=model_name,
        gradient_checkpointing=False,
    ).to(DEVICE)

    sd = torch.load(os.path.join(model_dir, f"exp_{exp_name}_best_model.pt"), map_location=DEVICE)
    model_bl.load_state_dict(sd)
    model_bl.eval()

    # Build dev dataloader with keyword features
    kw_feats = np.array(
        [extract_keyword_feature(kw, kw_to_idx) for kw in dev_kw_df["keyword"]],
        dtype=np.float32,
    )
    dev_ds = PCLDataset(
        texts=dev_kw_df["text"].tolist(),
        labels=dev_kw_df["binary_label"].tolist(),
        max_length=MAX_LENGTH,
        tokeniser=tokeniser_bl,
        extra_features=torch.tensor(kw_feats).to(DEVICE),
    )
    loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)
    metrics = evaluate(model_bl, DEVICE, loader, threshold=threshold)

    LOG.info(f"Baseline {exp_name}: F1={metrics['f1']:.4f}, P={metrics['precision']:.4f}, "
             f"R={metrics['recall']:.4f} (threshold={threshold:.3f})")

    del model_bl
    gc.collect(); torch.cuda.empty_cache()
    return metrics["preds"]


@torch.no_grad()
def generate_verbalizer_baseline_preds(
    exp_name: str, model_name: str, params_dir: str, model_dir: str,
    dev_df_plain: pd.DataFrame,
) -> np.ndarray:
    """Load a PCLVerbalizer baseline and return dev predictions."""
    cfg = load_baseline_config(params_dir, exp_name)
    threshold = cfg["best_threshold"]
    tokeniser_bl = AutoTokenizer.from_pretrained(model_name)

    # Same VERBALIZER_SETS / TEMPLATE_OPTIONS as defined in the experiments
    verb_set = [v for v in VERBALIZER_SETS if v[0] == cfg["verbalizer"]][0]
    bl_pos_ids = [tokeniser_bl.encode(w, add_special_tokens=False)[0] for w in verb_set[1]]
    bl_neg_ids = [tokeniser_bl.encode(w, add_special_tokens=False)[0] for w in verb_set[2]]
    bl_template = TEMPLATE_OPTIONS[cfg["template_idx"]]

    cache_dir = "/vol/bitbucket/jc4423/.cache/huggingface" if os.getlogin() == "jc4423" else None
    model_bl = PCLVerbalizer(
        pos_verbalizer_ids=bl_pos_ids,
        neg_verbalizer_ids=bl_neg_ids,
        model_name=model_name,
        gradient_checkpointing=False,
        cache_dir=cache_dir,
    ).to(DEVICE)

    sd = torch.load(os.path.join(model_dir, f"exp_{exp_name}_best_model.pt"), map_location=DEVICE)
    model_bl.load_state_dict(sd)
    model_bl.eval()

    dev_ds = PCLVerbalizerDataset(
        texts=dev_df_plain["text"].tolist(),
        labels=dev_df_plain["binary_label"].tolist(),
        max_length=MAX_LENGTH,
        tokeniser=tokeniser_bl,
        template=bl_template,
    )
    loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)
    metrics = evaluate(model_bl, DEVICE, loader, threshold=threshold)

    LOG.info(f"Baseline {exp_name}: F1={metrics['f1']:.4f}, P={metrics['precision']:.4f}, "
             f"R={metrics['recall']:.4f} (threshold={threshold:.3f})")

    del model_bl
    gc.collect(); torch.cuda.empty_cache()
    return metrics["preds"]


# --- Generate baseline predictions ---
# dev_df already has 'keyword' column from load_data_with_keyword
baseline_clf_preds = generate_classifier_baseline_preds(
    BASELINE_CLASSIFIER_EXP, BASELINE_CLASSIFIER_MODEL,
    BASELINE_CLASSIFIER_PARAMS_DIR, BASELINE_CLASSIFIER_MODEL_DIR,
    dev_df,  # needs keyword column
)
baseline_verb_preds = generate_verbalizer_baseline_preds(
    BASELINE_VERBALIZER_EXP, BASELINE_VERBALIZER_MODEL,
    BASELINE_VERBALIZER_PARAMS_DIR, BASELINE_VERBALIZER_MODEL_DIR,
    dev_df,
)

dev_df["baseline_clf_pred"] = baseline_clf_preds
dev_df["baseline_verb_pred"] = baseline_verb_preds

In [ ]:
def compare_against_baseline(dev_df: pd.DataFrame, baseline_col: str, baseline_label: str):
    """Cross-tabulate our model vs a baseline and print summary + examples."""
    bl_correct = (dev_df[baseline_col] == dev_df["binary_label"]).astype(int)

    crosstab = pd.crosstab(
        pd.Categorical(dev_df["correct"].map({1: "Ours ✓", 0: "Ours ✗"}),
                       categories=["Ours ✓", "Ours ✗"]),
        pd.Categorical(bl_correct.map({1: f"{baseline_label} ✓", 0: f"{baseline_label} ✗"}),
                       categories=[f"{baseline_label} ✓", f"{baseline_label} ✗"]),
        margins=True,
    )
    print(f"\n{'='*60}")
    print(f" Cross-tabulation: Ours vs {baseline_label}")
    print(f"{'='*60}")
    print(crosstab)

    both_right = ((dev_df["correct"] == 1) & (bl_correct == 1)).sum()
    both_wrong = ((dev_df["correct"] == 0) & (bl_correct == 0)).sum()
    ours_only  = ((dev_df["correct"] == 1) & (bl_correct == 0)).sum()
    base_only  = ((dev_df["correct"] == 0) & (bl_correct == 1)).sum()

    print(f"\n  Both correct:                {both_right}")
    print(f"  Both wrong:                  {both_wrong}")
    print(f"  Only ours correct:           {ours_only}")
    print(f"  Only {baseline_label} correct: {base_only}")

    # Show examples where models disagree
    ours_wins = dev_df[(dev_df["correct"] == 1) & (bl_correct == 0)]
    base_wins = dev_df[(dev_df["correct"] == 0) & (bl_correct == 1)]
    show_samples(ours_wins, f"OURS CORRECT, {baseline_label} WRONG", n=8, sort_col="score", ascending=False)
    show_samples(base_wins, f"{baseline_label} CORRECT, OURS WRONG", n=8, sort_col="score", ascending=False)


# --- Compare against classifier baseline (exp_A) ---
compare_against_baseline(dev_df, "baseline_clf_pred", BASELINE_CLASSIFIER_EXP)

# --- Compare against verbalizer baseline (exp_L) ---
compare_against_baseline(dev_df, "baseline_verb_pred", BASELINE_VERBALIZER_EXP)

## 4. Confusion Matrix & Precision–Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion Matrix ---
cm = confusion_matrix(labels_np, preds_np)
disp = ConfusionMatrixDisplay(cm, display_labels=["Non-PCL", "PCL"])
disp.plot(ax=axes[0], cmap="Blues", values_format="d")
axes[0].set_title(f"Confusion Matrix (threshold={BEST_THRESHOLD:.3f})")

# --- Precision–Recall Curve ---
precision_arr, recall_arr, thresholds_pr = precision_recall_curve(labels_np, probs)
ap = average_precision_score(labels_np, probs)
axes[1].plot(recall_arr, precision_arr, linewidth=2, label=f"AP = {ap:.3f}")
axes[1].axhline(y=labels_np.mean(), color="grey", linestyle="--", label="Prevalence")

# Mark the operating threshold
op_idx = np.argmin(np.abs(thresholds_pr - BEST_THRESHOLD))
axes[1].scatter([recall_arr[op_idx]], [precision_arr[op_idx]], s=120, c="red", zorder=5,
                label=f"Operating point (t={BEST_THRESHOLD:.2f})")

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall Curve")
axes[1].legend()
axes[1].set_xlim(0, 1.02)
axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_analysis_cm_pr.png", dpi=300)
plt.show()

In [ ]:
# --- ROC Curve ---
fpr, tpr, thresholds_roc = roc_curve(labels_np, probs)
auc = roc_auc_score(labels_np, probs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="grey", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_analysis_roc.png", dpi=300)
plt.show()

print(f"AUC-ROC:          {auc:.4f}")
print(f"Average Precision: {ap:.4f}")

## 5. Ablation: Threshold Sensitivity

Sweep thresholds to show F1/precision/recall trade-offs.

In [ ]:
thresholds = np.arange(0.10, 0.90, 0.01)
f1s, precs, recs = [], [], []
for t in thresholds:
    p = (probs >= t).astype(int)
    f1s.append(f1_score(labels_np, p, zero_division=0))
    precs.append(precision_score(labels_np, p, zero_division=0))
    recs.append(recall_score(labels_np, p, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, f1s, label="F1", linewidth=2)
ax.plot(thresholds, precs, label="Precision", linewidth=1.5, linestyle="--")
ax.plot(thresholds, recs, label="Recall", linewidth=1.5, linestyle="--")
ax.axvline(x=BEST_THRESHOLD, color="red", linestyle=":", label=f"Best t={BEST_THRESHOLD:.2f}")

best_sweep_idx = np.argmax(f1s)
ax.scatter([thresholds[best_sweep_idx]], [f1s[best_sweep_idx]], s=100, c="green", zorder=5,
           label=f"Peak F1={f1s[best_sweep_idx]:.4f} @ t={thresholds[best_sweep_idx]:.2f}")

ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Sensitivity Analysis")
ax.legend()
ax.set_xlim(0.10, 0.90)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_analysis_threshold.png", dpi=300)
plt.show()

## 6. Per-Keyword Breakdown

Analyse performance by the **keyword** (vulnerable group) associated
with each paragraph (e.g. 'homeless', 'migrant', 'disabled').

In [ ]:
kw_stats = []
for kw, group in dev_df.groupby("keyword"):
    n = len(group)
    n_pos = group["binary_label"].sum()
    if n_pos == 0:
        # Can't compute F1 for keyword with no positive samples
        kw_stats.append({"keyword": kw, "n": n, "n_pos": n_pos,
                         "f1": np.nan, "precision": np.nan, "recall": np.nan,
                         "fp": (group["error_type"] == "FP").sum(),
                         "fn": 0})
        continue
    f1 = f1_score(group["binary_label"], group["pred"], zero_division=0)
    prec = precision_score(group["binary_label"], group["pred"], zero_division=0)
    rec = recall_score(group["binary_label"], group["pred"], zero_division=0)
    kw_stats.append({"keyword": kw, "n": n, "n_pos": n_pos,
                     "f1": f1, "precision": prec, "recall": rec,
                     "fp": (group["error_type"] == "FP").sum(),
                     "fn": (group["error_type"] == "FN").sum()})

kw_df = pd.DataFrame(kw_stats).sort_values("f1", ascending=True)
print(kw_df.to_string(index=False))

In [ ]:
# Bar chart of per-keyword F1
valid_kw = kw_df.dropna(subset=["f1"])
cmap = plt.colormaps["RdYlGn"]
fig, ax = plt.subplots(figsize=(10, max(4, len(valid_kw) * 0.4)))
colors = cmap(valid_kw["f1"].values)  # Red = low F1, green = high
ax.barh(valid_kw["keyword"], valid_kw["f1"], color=colors)
ax.set_xlabel("F1 Score")
ax.set_title("Per-Keyword F1 on Dev Set")
ax.set_xlim(0, 1)
for i, (_, row) in enumerate(valid_kw.iterrows()):
    ax.text(row["f1"] + 0.01, i, f"{row['f1']:.2f} (n={int(row['n_pos'])})",
            va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_analysis_keyword_f1.png", dpi=300)
plt.show()

## 7. Per-Category Breakdown

PCL paragraphs are annotated with one or more PCL categories
(e.g. `Authority_voice`, `Compassion`, `Shallow_solution`).
This reveals which *types* of patronising language the model catches.

In [ ]:
cat_stats = []
for cat in PCL_CATEGORIES:
    cat_samples = dev_df[dev_df[cat] == 1]
    n = len(cat_samples)
    if n == 0:
        continue
    tp = ((cat_samples["pred"] == 1) & (cat_samples["binary_label"] == 1)).sum()
    fn = ((cat_samples["pred"] == 0) & (cat_samples["binary_label"] == 1)).sum()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    cat_stats.append({"category": cat, "n": n, "recall": recall, "tp": tp, "fn": fn})

cat_df = pd.DataFrame(cat_stats).sort_values("recall", ascending=True)
print(cat_df.to_string(index=False))

# Visualise
cmap = plt.colormaps["RdYlGn"]
fig, ax = plt.subplots(figsize=(9, 4))
colors = cmap(cat_df["recall"].values)
bars = ax.barh(cat_df["category"], cat_df["recall"], color=colors)
ax.set_xlabel("Recall")
ax.set_title("Per-Category Recall (among PCL paragraphs with that category)")
ax.set_xlim(0, 1.1)
for i, (_, row) in enumerate(cat_df.iterrows()):
    ax.text(row["recall"] + 0.01, i, f"{row['recall']:.2f} (n={row['n']})",
            va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_analysis_category_recall.png", dpi=300)
plt.show()

## 8. Error vs Text Length & Confidence Analysis

Investigate whether the model struggles with short/long texts,
and analyse the confidence distribution of errors.

In [ ]:
dev_df["text_len"] = dev_df["text"].str.len()
dev_df["word_count"] = dev_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Text length distribution by error type ---
for etype, color in [("TP", "green"), ("TN", "lightblue"), ("FP", "orange"), ("FN", "red")]:
    subset = dev_df[dev_df["error_type"] == etype]
    axes[0].hist(subset["word_count"], bins=30, alpha=0.5, label=f"{etype} (n={len(subset)})",
                 color=color, density=True)
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Density")
axes[0].set_title("Word Count Distribution by Error Type")
axes[0].legend()

# --- Score distribution by error type ---
for etype, color in [("TP", "green"), ("TN", "lightblue"), ("FP", "orange"), ("FN", "red")]:
    subset = dev_df[dev_df["error_type"] == etype]
    axes[1].hist(subset["score"], bins=40, alpha=0.5, label=f"{etype} (n={len(subset)})",
                 color=color, density=True)
axes[1].axvline(x=BEST_THRESHOLD, color="black", linestyle=":", label=f"Threshold={BEST_THRESHOLD:.2f}")
axes[1].set_xlabel("Model Score (sigmoid probability)")
axes[1].set_ylabel("Density")
axes[1].set_title("Score Distribution by Error Type")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/error_analysis_length_confidence.png", dpi=300)
plt.show()

In [ ]:
# --- Summary statistics of errors ---
summary = dev_df.groupby("error_type").agg(
    count=("text", "count"),
    mean_word_count=("word_count", "mean"),
    median_word_count=("word_count", "median"),
    mean_score=("score", "mean"),
    std_score=("score", "std"),
).round(2)
print(summary)

## 9. Hardest Samples — Confident Errors

The most informative errors are those where the model is *confident*
but wrong. These reveal systematic biases.

In [ ]:
# Most confidently wrong FPs (highest score, but label=0)
show_samples(
    fp_df, "MOST CONFIDENT FALSE POSITIVES (model sure it's PCL, but it's not)",
    n=10, sort_col="score", ascending=False,
)

# Most confidently wrong FNs (lowest score, but label=1)
show_samples(
    fn_df, "MOST CONFIDENT FALSE NEGATIVES (model sure it's NOT PCL, but it is)",
    n=10, sort_col="score", ascending=True,
)

## 10. Additional Analysis — Skeleton

Add any further custom analysis below. Some ideas:
- N-gram analysis of FP/FN texts
- Comparing verbalizer variants on the same data
- Ablation: removing augmentation / focal loss
- Cross-experiment comparison table

In [ ]:
# ============================================================
# SKELETON: Cross-experiment comparison table
# ============================================================
# Fill in results from your different experiments for a summary table.
# fmt: experiment_name -> {"dev_f1": ..., "dev_p": ..., "dev_r": ...}

experiment_results = {
    # "A_baseline":       {"dev_f1": 0.00, "dev_p": 0.00, "dev_r": 0.00},
    # "B_focal":          {"dev_f1": 0.00, "dev_p": 0.00, "dev_r": 0.00},
    # "E_topk_zscore":    {"dev_f1": 0.00, "dev_p": 0.00, "dev_r": 0.00},
    # "H_augmentation":   {"dev_f1": 0.00, "dev_p": 0.00, "dev_r": 0.00},
    # "O_verbalizer":     {"dev_f1": 0.00, "dev_p": 0.00, "dev_r": 0.00},
}

if experiment_results:
    comp_df = pd.DataFrame(experiment_results).T
    comp_df.index.name = "experiment"
    comp_df = comp_df.sort_values("dev_f1", ascending=False)
    print(comp_df.to_string())
else:
    print("No experiment results filled in yet. Edit the dict above.")

In [ ]:
# ============================================================
# SKELETON: N-gram analysis of errors
# ============================================================
from collections import Counter

def top_ngrams(texts: list[str], n: int = 2, top_k: int = 20) -> list[tuple]:
    """Extract most common n-grams from a list of texts."""
    counter = Counter()
    for text in texts:
        words = text.lower().split()
        for i in range(len(words) - n + 1):
            counter[tuple(words[i:i+n])] += 1
    return counter.most_common(top_k)


print("Top bigrams in FALSE NEGATIVES:")
for ngram, count in top_ngrams(fn_df["text"].tolist(), n=2, top_k=15):
    print(f"  {' '.join(ngram):<30} {count}")

print("\nTop bigrams in FALSE POSITIVES:")
for ngram, count in top_ngrams(fp_df["text"].tolist(), n=2, top_k=15):
    print(f"  {' '.join(ngram):<30} {count}")

In [ ]:
# ============================================================
# SKELETON: Export errors for manual annotation
# ============================================================

# Export all errors to CSV for easy manual review
errors_df = dev_df[dev_df["error_type"].isin(["FP", "FN"])].copy()
errors_df = errors_df[["text", "keyword", "binary_label", "pred", "score", "error_type"]]
errors_df.to_csv(f"{OUT_DIR}/error_analysis_errors.csv", index=True)
LOG.info(f"Exported {len(errors_df)} errors to {OUT_DIR}/error_analysis_errors.csv")